# Tầng 2 — cho điểm từng câu bằng checkpoint đã huấn luyện

Notebook này **không huấn luyện gì**. Nó nạp checkpoint tầng 2 từ output của kernel
`dl-summarisevn-tang2` (khai trong `kernel_sources`), chạy
`src/models/phobert_select.py score` trên `tune` và `val`, rồi đóng gói hai file điểm.
Bước chọn quy tắc (`select`) không cần GPU và chạy trên máy.

Có notebook này vì tải checkpoint 540 MB về máy bị đứt kết nối nhiều lần; cho điểm
trên Kaggle chỉ tốn vài phút GPU và không phải chuyển trọng số đi đâu cả.

## Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `GPU T4 x2` (notebook tự ghim còn một) |
| **Internet** | `On` |

In [ ]:
# ==== CHI SUA O NAY ===================================================
SPLITS = ["tune", "val"]
CKPT_GLOB = "/kaggle/input/**/phobert_sent/final"
# ======================================================================

REPO = "https://github.com/ICY825/SummariseVietNamese.git"
DIR = "/kaggle/working/BTL_DL"


def check(code, what):
    """Dung notebook neu lenh `!` ngay truoc do loi — `!lenh` loi khong nem ngoai le."""
    if code != 0:
        raise RuntimeError(f"{what} THAT BAI (ma thoat {code}) -- xem log ngay tren.")
    print(f"{what}: OK")


print("Cho diem tang 2 tren", SPLITS)

In [ ]:
import torch
print("GPU thay duoc:", torch.cuda.device_count())
if torch.cuda.device_count() == 0:
    raise RuntimeError("Khong thay GPU. Settings > Accelerator > GPU T4 x2.")

In [ ]:
import os
if not os.path.isdir(DIR):
    !git clone -q {REPO} {DIR}
    check(_exit_code, "git clone")
os.chdir(DIR)
!git pull -q
check(_exit_code, "git pull")
!git log --oneline -1
!bash -c "set -o pipefail; python src/models/selftest.py | tail -2"
check(_exit_code, "selftest")

In [ ]:
import glob

found = sorted(glob.glob(CKPT_GLOB, recursive=True))
found = [f for f in found if os.path.exists(os.path.join(f, "head.pt"))]
if not found:
    raise RuntimeError(
        f"Khong thay checkpoint co head.pt khop {CKPT_GLOB}. Gan output cua "
        "dl-summarisevn-tang2 lam input.\n"
        f"Hien /kaggle/input co: {sorted(glob.glob('/kaggle/input/*/*'))[:20]}")
CKPT = found[0]
print("Checkpoint:", CKPT)
!ls -la {CKPT}

In [ ]:
for split in SPLITS:
    cmd = f"CUDA_VISIBLE_DEVICES=0 python src/models/phobert_select.py score --model {CKPT} --split {split}"
    print(cmd)
    !{cmd}
    check(_exit_code, f"cho diem {split}")

In [ ]:
import pathlib
import zipfile

picked = sorted(pathlib.Path("results/predictions").glob("phobert-sent-*_scores.json"))
if len(picked) != len(SPLITS):
    raise RuntimeError(f"Mong {len(SPLITS)} file diem, thay {len(picked)}: {picked}")
zpath = "/kaggle/working/ket_qua_tang2_score.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in picked:
        z.write(p, p.as_posix())
        print(f"  {p.as_posix():80s} {p.stat().st_size / 1e6:6.2f} MB")
print("Da dong goi:", zpath)

## Sau khi chạy

Tải `ket_qua_tang2_score.zip`, giải nén tại gốc repo, rồi trên máy:

    .venv/Scripts/python.exe src/models/phobert_select.py select